In [7]:
import os

# Set the correct Java Home
os.environ['JAVA_HOME'] = "/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home"

# Set the Spark Home
os.environ['SPARK_HOME'] = "/opt/homebrew/opt/apache-spark/libexec"

In [1]:
import pandas as pd

In [2]:
data = [[1, 'Wang', 'Allen'], [2, 'Alice', 'Bob']]
person = pd.DataFrame(data, columns=['personId', 'firstName', 'lastName']).astype({'personId':'Int64', 'firstName':'object', 'lastName':'object'})
data = [[1, 2, 'New York City', 'New York'], [2, 3, 'Leetcode', 'California']]
address = pd.DataFrame(data, columns=['addressId', 'personId', 'city', 'state']).astype({'addressId':'Int64', 'personId':'Int64', 'city':'object', 'state':'object'})

In [3]:
person

,personId,firstName,lastName
0,1,Wang,Allen
1,2,Alice,Bob


In [4]:
address

,addressId,personId,city,state
0,1,2,New York City,New York
1,2,3,Leetcode,California


In [5]:
from pyspark.sql import SparkSession

In [6]:
spark = SparkSession.builder.appName("combineTables").getOrCreate()
print(spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/10/20 18:10:51 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/10/20 18:10:51 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Py4JError: An error occurred while calling None.org.apache.spark.sql.SparkSession. Trace:
py4j.Py4JException: Constructor org.apache.spark.sql.SparkSession([class org.apache.spark.SparkContext, class java.util.HashMap]) does not exist
	at py4j.reflection.ReflectionEngine.getConstructor(ReflectionEngine.java:180)
	at py4j.reflection.ReflectionEngine.getConstructor(ReflectionEngine.java:197)
	at py4j.Gateway.invoke(Gateway.java:237)
	at py4j.commands.ConstructorCommand.invokeConstructor(ConstructorCommand.java:80)
	at py4j.commands.ConstructorCommand.execute(ConstructorCommand.java:69)
	at py4j.ClientServerConnection.waitForCommands(ClientServerConnection.java:184)
	at py4j.ClientServerConnection.run(ClientServerConnection.java:108)
	at java.base/java.lang.Thread.run(Thread.java:1583)



In [14]:
df_person = spark.createDataFrame(person)
df_address = spark.createDataFrame(address)


In [15]:
df_person.show()

+--------+---------+--------+
|personId|firstName|lastName|
+--------+---------+--------+
|       1|     Wang|   Allen|
|       2|    Alice|     Bob|
+--------+---------+--------+



In [16]:
df_address.show()

+---------+--------+-------------+----------+
|addressId|personId|         city|     state|
+---------+--------+-------------+----------+
|        1|       2|New York City|  New York|
|        2|       3|     Leetcode|California|
+---------+--------+-------------+----------+



In [19]:
result = df_person.join(df_address,df_person.personId == df_address.personId,"left") \
                  .select("firstName","lastname","city","state")

result.show()

+---------+--------+-------------+--------+
|firstName|lastname|         city|   state|
+---------+--------+-------------+--------+
|     Wang|   Allen|         NULL|    NULL|
|    Alice|     Bob|New York City|New York|
+---------+--------+-------------+--------+



25/10/11 23:18:15 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 903154 ms exceeds timeout 120000 ms
25/10/11 23:18:15 WARN SparkContext: Killing executors is not supported by current scheduler.
25/10/11 23:18:23 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:342)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$